<a href="https://colab.research.google.com/github/thisishasan/speech_processing/blob/main/02_generated_data_conversion_to_llava_compatible_format.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json
import os
import shutil
from collections import Counter

from google.colab import drive

DRIVE_ROOT = "/content/drive/MyDrive/01_speech_processing/sp_exam_project/dataset"
INPUT_DIR = os.path.join(DRIVE_ROOT, "outputs")
IMAGE_DIR = os.path.join(DRIVE_ROOT, "images")
OUTPUT_DIR = os.path.join(DRIVE_ROOT, "llava_dataset")

IMAGE_FIELD = "filename"
COPY_IMAGES = False
SKIP_MISSING_IMAGES = True

SPLITS = ("train", "val", "test")

def read_json(path):
    with open(path, "r", encoding="utf-8") as file:
        data = json.load(file)
    if not isinstance(data, list):
        raise ValueError(f"Expected a JSON list in {path}")
    return data


def clean_text(value):
    return " ".join(str(value or "").split()).strip()


def image_path_for(record):
    filename = clean_text(record.get("filename"))
    original_path = clean_text(record.get("image"))

    if filename:
        configured_path = os.path.join(IMAGE_DIR, filename)
        if os.path.exists(configured_path):
            return configured_path, filename

    if original_path and os.path.exists(original_path):
        return original_path, filename or os.path.basename(original_path)

    return configured_path if filename else original_path, filename


def make_llava_record(record, split, index):
    question = clean_text(record.get("question"))
    answer = clean_text(record.get("answer"))
    filename = clean_text(record.get("filename"))

    if not question or not answer or not filename:
        return None, "missing question, answer, or filename"

    image_path, resolved_filename = image_path_for(record)
    if SKIP_MISSING_IMAGES and not os.path.exists(image_path):
        return None, f"image not found: {image_path}"

    if IMAGE_FIELD == "absolute_path":
        image_value = image_path
    else:
        image_value = resolved_filename

    if question.lower().startswith("<image>"):
        human_value = question
    else:
        human_value = f"<image>\n{question}"

    source_id = clean_text(record.get("id"))
    sample_id = source_id or f"{split}_{index:07d}"

    return {
        "id": sample_id,
        "image": image_value,
        "conversations": [
            {"from": "human", "value": human_value},
            {"from": "gpt", "value": answer},
        ],
    }, None


def write_json(path, data):
    with open(path, "w", encoding="utf-8") as file:
        json.dump(data, file, ensure_ascii=False, indent=2)

drive.mount("/content/drive")
os.makedirs(OUTPUT_DIR, exist_ok=True)

converted_by_split = {}
missing = []
seen = set()

for split in SPLITS:
    input_path = os.path.join(INPUT_DIR, f"vqain_{split}.json")
    if not os.path.exists(input_path):
        print(f"{split.upper()}: input file not found: {input_path}")
        converted_by_split[split] = []
        continue

    records = read_json(input_path)
    converted = []

    for index, record in enumerate(records):
        llava_record, error = make_llava_record(record, split, index)
        if error:
            missing.append({
                "split": split,
                "filename": record.get("filename", ""),
                "error": error,
            })
            continue

        key = (
            llava_record["image"],
            llava_record["conversations"][0]["value"].lower(),
            llava_record["conversations"][1]["value"].lower(),
        )
        if key in seen:
            continue
        seen.add(key)
        converted.append(llava_record)

    converted_by_split[split] = converted
    write_json(os.path.join(OUTPUT_DIR, f"llava_{split}.json"), converted)

    print(f"{split.upper()}:")
    print(f"  Input QA pairs: {len(records):,}")
    print(f"  LLaVA records: {len(converted):,}")
    print(f"  Unique images: {len({x['image'] for x in converted}):,}")


all_records = [record for split in SPLITS for record in converted_by_split[split]]
write_json(os.path.join(OUTPUT_DIR, "llava_all.json"), all_records)
write_json(os.path.join(OUTPUT_DIR, "missing_images.json"), missing)

summary = {
    "input_dir": INPUT_DIR,
    "image_dir": IMAGE_DIR,
    "output_dir": OUTPUT_DIR,
    "image_field": IMAGE_FIELD,
    "copy_images": COPY_IMAGES,
    "records_by_split": {
        split: len(converted_by_split[split]) for split in SPLITS
    },
    "unique_images_by_split": {
        split: len({record["image"] for record in converted_by_split[split]})
        for split in SPLITS
    },
    "qa_records_total": len(all_records),
    "unique_images_total": len({record["image"] for record in all_records}),
    "skipped_records": len(missing),
    "skipped_by_reason": dict(Counter(item["error"] for item in missing)),
}
write_json(os.path.join(OUTPUT_DIR, "conversion_summary.json"), summary)

if COPY_IMAGES:
    copied_dir = os.path.join(OUTPUT_DIR, "images")
    for record in all_records:
        source = os.path.join(IMAGE_DIR, record["image"])
        destination = os.path.join(copied_dir, record["image"])
        os.makedirs(os.path.dirname(destination), exist_ok=True)
        if os.path.exists(source) and not os.path.exists(destination):
            shutil.copy2(source, destination)
    print(f"Images copied to: {copied_dir}")

print("\ncompleted: ")
print(json.dumps(summary, indent=2))
print(f"\nLLaVA dataset files saved in: {OUTPUT_DIR}")


Mounted at /content/drive
TRAIN:
  Input QA pairs: 9,294
  LLaVA records: 9,294
  Unique images: 2,767
VAL:
  Input QA pairs: 357
  LLaVA records: 357
  Unique images: 101
TEST:
  Input QA pairs: 356
  LLaVA records: 356
  Unique images: 106

completed: 
{
  "input_dir": "/content/drive/MyDrive/01_speech_processing/sp_exam_project/dataset/outputs",
  "image_dir": "/content/drive/MyDrive/01_speech_processing/sp_exam_project/dataset/images",
  "output_dir": "/content/drive/MyDrive/01_speech_processing/sp_exam_project/dataset/llava_dataset",
  "image_field": "filename",
  "copy_images": false,
  "records_by_split": {
    "train": 9294,
    "val": 357,
    "test": 356
  },
  "unique_images_by_split": {
    "train": 2767,
    "val": 101,
    "test": 106
  },
  "qa_records_total": 10007,
  "unique_images_total": 2974,
  "skipped_records": 0,
  "skipped_by_reason": {}
}

LLaVA dataset files saved in: /content/drive/MyDrive/01_speech_processing/sp_exam_project/dataset/llava_dataset


In [2]:
!ls -la $OUTPUT_DIR

total 5335
drwx------ 2 root root    4096 Sep 13 05:31 .
drwx------ 6 root root    4096 Sep 13 05:29 ..
-rw------- 1 root root     620 Sep 13 05:31 conversion_summary.json
-rw------- 1 root root 2726339 Sep 13 05:31 llava_all.json
-rw------- 1 root root   96801 Sep 13 05:31 llava_test.json
-rw------- 1 root root 2533343 Sep 13 05:31 llava_train.json
-rw------- 1 root root   96199 Sep 13 05:31 llava_val.json
-rw------- 1 root root       2 Sep 13 05:31 missing_images.json
